# Tarstrade — tars-lora out-of-sample eval kernel

Purpose: run the trained QLoRA carry judge (`Henoch4/tars-lora` adapter) on the REAL
out-of-sample rows of `data/carry/dataset.csv`, simulate the trading policy, and run
the LIVE gate (`src/validation.py`, vendored) — the honest check before any wiring
into the live agent (Step 1 of the integration plan).

## What you must understand before reading results

- **The OOS window is a dead-funding regime.** The test split is the 2026 tail of the
  walk-forward fold (Feb 2025–Aug 2026 tails). Realized 7-day carry (`y_sum_7d_bps`)
  never clears the label bar of 2×35 = **70 bps** there — `y_win` fires on ZERO test
  rows. The correct strategy in that window is to NOT trade.
- The repo's gate (validation.py:154) requires `has_oos_evidence` (a nontrivial OOS
  return stream) plus OOS Calmar >= 1.0. A policy that never trades therefore CANNOT
  clear the gate. That is the correct evidence-based FAIL — not a bug.
- So this kernel produces **two layers of truth**:
  1. the gate verdict (`cleared_for_paper_trading`) for the model policy, AND
  2. the signal diagnostics — does the model's yes/no *score* rank realized 7d carry
     at all (Spearman, decile table, top vs bottom)? Even in a no-winner regime,
     ranking ability is what would matter when a carry regime returns.
- The incumbent constant gate (`f0 <= -0.001`, src/agent.py:442) is simulated on the
  same rows for the honest comparison: replacing a *bleeding* constant rule with a
  *sitting-out* model is still a win for the model even though promotion is denied.

In [ ]:
# Cell 1 — GPU check + FIX: pin to one T4 BEFORE torch imports (T4 x2 spreads
# the model and crashes the embedding index_select — same bug as training Cell 4).
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
!nvidia-smi
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# Cell 2 — Install (eval is inference-only; needs the bnb 4-bit loader)
!pip install -q transformers peft accelerate bitsandbytes safetensors numpy pandas
print("installed")

In [ ]:
# Cell 3 — Flexibly locate the tars-eval Input (resilient to the mount depth;
# Kaggle mounts datasets as /kaggle/input/<slug>/ or /kaggle/input/<owner>/<slug>/)
import glob, pathlib, shutil, sys

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

def find(name: str):
    c = list(glob.glob(f'/kaggle/input/**/{name}', recursive=True)) + \
        [str(p) for p in pathlib.Path('.').glob(name)]
    c = list(dict.fromkeys(c))
    if not c:
        raise SystemExit(f'{name} not found — upload the tars-eval Dataset as an Input')
    return c[0]

dataset_csv = find('dataset.csv')
sys.path.insert(0, str(pathlib.Path(dataset_csv).parent))
import validation, eval_lib
adapter_dir = None
for cand in glob.glob('/kaggle/input/**/lora_model', recursive=True):
    if pathlib.Path(cand).is_dir():
        adapter_dir = cand; break
if not adapter_dir:
    raise SystemExit('lora_model/ not found in tars-eval dataset')
print('dataset.csv :', dataset_csv)
print('adapter     :', adapter_dir)
print('validation.py + eval_lib.py import OK — these are the SAME bytes verified locally')

In [ ]:
# Cell 4 — Load the real OOS rows and render the training-format prompts
import pandas as pd
from eval_lib import (select_eval_rows, build_prompts, FEATURES)

df = pd.read_csv(dataset_csv)
rows = select_eval_rows(df)
prompts = build_prompts(rows)
print('dataset rows      :', len(df))
print('OOS eval rows     :', len(rows), ' (test & usable & 6 features present)')
print('y_win positives   :', int(rows['y_win'].sum()), '(expected 0 in the dead-funding window)')
print('max y_sum_7d_bps  :', round(float(rows['y_sum_7d_bps'].max()), 2), '(label bar:', 70, 'bps)')
print('sample prompt     :', prompts[0])
assert int(rows['y_win'].sum()) == 0, 'test window unexpectedly has y_win positives — re-check splits'

In [ ]:
# Cell 5 — Load the SAME 4-bit base as training + the LoRA adapter
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()
print('adapter applied on 4-bit base, device:', model.device)

# Yes/No token ids for first-token scoring
yes_ids, no_ids = set(), set()
for t in ["yes", " yes", "Yes", "YES"]:
    yes_ids.update(tokenizer(t, add_special_tokens=False).input_ids)
for t in ["no", " no", "No", "NO"]:
    no_ids.update(tokenizer(t, add_special_tokens=False).input_ids)
print('yes token ids:', sorted(yes_ids), ' no token ids:', sorted(no_ids))

In [ ]:
# Cell 6 — Score every OOS row. Score = P(yes) on the FIRST decision token
# (greedy argmax == binary decision; logit gives the continuous signal the
# diagnostics need). No full generation needed for 49k rows — fast on T4.
import numpy as np
import torch

BATCH = 64
scores = np.empty(len(prompts), dtype=np.float64)
yes_ids_l = list(yes_ids); no_ids_l = list(no_ids)

for s in range(0, len(prompts), BATCH):
    chunk = prompts[s:s + BATCH]
    enc = tokenizer(chunk, padding=True, truncation=True, max_length=512,
                    return_tensors="pt")
    with torch.no_grad():
        logits = model(enc.input_ids.to(model.device),
                       attention_mask=enc.attention_mask.to(model.device)).logits[:, -1, :]
    p = torch.softmax(logits.float(), dim=-1)
    p_yes = p[:, yes_ids_l].sum(-1)
    p_no = p[:, no_ids_l].sum(-1)
    scores[s:s + BATCH] = (p_yes / (p_yes + p_no + 1e-12)).cpu().numpy()

rows['score'] = scores
rows['enter'] = eval_lib.binary_from_score(scores)
np.savetxt('/kaggle/working/scores.csv',
           np.column_stack([rows['ts'].to_numpy(), scores]), delimiter=',',
           header='ts,score', comments='')
print('scored', len(scores), 'rows in', round(len(prompts) / BATCH), 'batches')
print('score p10/p50/p90:', scores.min(), round(float(np.quantile(scores, 0.1)), 3),
      round(float(np.quantile(scores, 0.5)), 3), round(float(np.quantile(scores, 0.9)), 3)
      if len(scores) else None)
print('enter rate:', round(100.0 * float(rows['enter'].mean()), 3), '%')

In [ ]:
# Cell 7 — QA cross-check: full greedy generation on a sample; the parse path
# must agree with the logit decision (and give you readable example answers).
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(prompts), size=min(300, len(prompts)), replace=False)
sample_idx.sort()
agree, unparsed = 0, 0
examples = []
for i in sample_idx:
    enc = tokenizer(prompts[i], return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**enc.to(model.device), max_new_tokens=8,
                             temperature=0.0, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)[len(prompts[i]):]
    parsed = eval_lib.parse_generated(text)
    if parsed == "other":
        unparsed += 1
    agree += int((parsed == "yes") == bool(rows['enter'].iloc[i]))
    if len(examples) < 5:
        examples.append((prompts[i].split(' -> ')[0], parsed, round(float(rows['score'].iloc[i]), 3)))
print('logit-vs-generate agreement:', round(agree / len(sample_idx), 4),
      ' unparseable:', unparsed, '/', len(sample_idx))
for p, a, s in examples:
    print('  score', s, '->', a, '|', p)

In [ ]:
# Cell 8 — Policies: model (enter on score>=0.5) vs incumbent constant gate
# (f0 <= -0.001, src/agent.py:442) vs cash. Same rows, same cost model.
from eval_lib import (policy_net_bps, portfolio_series, trades_summary,
                      incumbent_enter, COST_BPS)

enter_model = rows['enter'].to_numpy()
enter_inc = incumbent_enter(rows)
y = rows['y_sum_7d_bps'].to_numpy()
print('label bar (2x cost):', 2 * COST_BPS, 'bps  | cost per round trip:', COST_BPS, 'bps')
print('\nMODEL policy:')
print(trades_summary(rows, enter_model))
print('\nINCUMBENT policy (|f0| gate as-is today):')
print(trades_summary(rows, enter_inc))
print('\nCASH policy: 0 trades, 0 bps')
_, s_model = portfolio_series(rows, policy_net_bps(rows, enter_model))
_, s_inc = portfolio_series(rows, policy_net_bps(rows, enter_inc))
print('\nportfolio periods:', len(s_model), ' model trades', int((enter_model).sum()),
      ' incumbent trades', int(enter_inc.sum()))

In [ ]:
# Cell 9 — The LIVE gate (vendored src/validation.py). Fractional returns are
# bps/10000; the gate's equity curve is multiplicative. Honest expectations:
#   * a model that never trades -> has_oos_evidence False -> cleared False
#   * an incumbent that bleeds  -> Calmar < 1 -> cleared False
from eval_lib import gate_report

r_model = gate_report(s_model)
r_inc = gate_report(s_inc)
print('MODEL policy gate:')
print(r_model)
print('\nINCUMBENT policy gate:')
print(r_inc)

In [ ]:
# Cell 10 — Signal diagnostics + cost sensitivity (evidence even with no winners)
from eval_lib import diagnostics, cost_sensitivity

diag = diagnostics(scores, y)
print('signal diagnostics:')
for k, v in diag.items():
    if k == 'deciles':
        print('  decile table (score -> mean realized 7d carry bps):')
        for d in v:
            print('   ', d)
    else:
        print(f'  {k}: {v}')
cs = cost_sensitivity(rows, enter_model)
print('\ncost sensitivity (model policy, total net bps):')
print(cs)

In [ ]:
# Cell 11 — Serialize the full evidence report (also saved to working dir)
import json, datetime

report = {
    'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'backend': 'gpu_t4_4bit_base',
    'adapter': 'Henoch4/tars-lora (fetched into tars-eval dataset)',
    'eval_rows': len(rows),
    'y_win_positives_in_eval': int(rows['y_win'].sum()),
    'label_bar_bps': 2 * COST_BPS,
    'model_policy': {
        'gate': r_model,
        'trades': trades_summary(rows, enter_model),
    },
    'incumbent_policy': {
        'gate': r_inc,
        'trades': trades_summary(rows, enter_inc),
    },
    'cash_policy': {'trades': 0, 'total_net_bps': 0.0},
    'diagnostics': diag,
    'cost_sensitivity': cs,
}
text = json.dumps(report, indent=2)
print(text)
open('/kaggle/working/tars-lora-eval-report.json', 'w').write(text)
print('\nsaved /kaggle/working/tars-lora-eval-report.json')

### How to read this honestly

- `cleared_for_paper_trading: false` is the expected, evidence-correct outcome in the
  dead-funding OOS window — promotion is DENIED until a carry regime reappears and
  the model shows rank + PnL edge there.
- The decision-relevant comparison is MODEL vs INCUMBENT on the same rows: the
  incumbent constant gate trades in the no-carry regime and bleeds; the model, if it
  sits out, is strictly better — that validates its role as a *veto* in shadow mode,
  not a promotion.
- Copy `tars-lora-eval-report.json` to `reports/tars-lora-oos-<date>.md` for the repo.

Regression safety: the eval lib and gate here are the SAME bytes as
`scripts/tars_lora_eval_lib.py` + `src/validation.py` (asserted in the dry-run).